# 02 — ArcFace for CEDAR Signature Verification

ArcFace replaces pair/triplet sampling with a **classification objective + angular margin**.  
Each writer is a class. The backbone learns embeddings that are angularly separable across writers.  
At inference, cosine similarity between L2-normalised embeddings is used for verification.

**Key differences  (contrastive):**

| | Contrastive (nb01) | ArcFace (nb02) |
|---|---|---|
| Training input | Pairs of images | Individual images + writer label |
| Loss | ContrastiveLoss on distance | Cross-entropy with angular margin |
| Similarity | Euclidean distance | Cosine similarity (L2-normalised) |
| Forgeries used in training | Yes | No (genuine only) |

---
## Section 0 — Setup

In [ ]:
import sys
import re
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn import metrics as sk_metrics
from sklearn.decomposition import PCA

def _find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "requirements.txt").exists():
            return p
    raise RuntimeError(f"Cannot find project root from {start}")

PROJECT_ROOT = _find_project_root(Path().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.data.transforms import get_default_transforms
from src.models.backbones import SmallCNN, ResNet18Embed
from src.metrics.verification import compute_metrics

SEED = 42
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device      :", DEVICE)
print("Project root:", PROJECT_ROOT)

---
## Section 1 — Load CEDAR

In [ ]:
DATA_ROOT = PROJECT_ROOT / "src" / "datasets" / "CEDAR"
ORG_DIR   = DATA_ROOT / "full_org"
FORG_DIR  = DATA_ROOT / "full_forg"

assert ORG_DIR.exists()  and FORG_DIR.exists(), f"CEDAR not found at {DATA_ROOT}"

PAT_ORG  = re.compile(r"^original_(\d+)_(\d+)\.png$",  re.IGNORECASE)
PAT_FORG = re.compile(r"^forgeries_(\d+)_(\d+)\.png$", re.IGNORECASE)

rows = []
for fp in ORG_DIR.iterdir():
    m = PAT_ORG.match(fp.name)
    if m:
        rows.append({"path": str(fp), "writer_id": int(m.group(1)),
                     "sample_id": int(m.group(2)), "label": "genuine"})
for fp in FORG_DIR.iterdir():
    m = PAT_FORG.match(fp.name)
    if m:
        rows.append({"path": str(fp), "writer_id": int(m.group(1)),
                     "sample_id": int(m.group(2)), "label": "forgery"})

df = pd.DataFrame(rows).sort_values(["writer_id", "label", "sample_id"]).reset_index(drop=True)
print(f"Total: {len(df)}  |  writers: {df['writer_id'].nunique()}")
print(df["label"].value_counts().to_string())

---
## Section 2 — Writer-level Split

In [ ]:
def split_writers(df, train=0.70, val=0.15, test=0.15, seed=42):
    writers = np.array(sorted(df["writer_id"].unique()))
    rng = np.random.default_rng(seed)
    rng.shuffle(writers)
    n = len(writers)
    n_train = int(round(n * train))
    n_val   = int(round(n * val))
    return set(writers[:n_train]), set(writers[n_train:n_train+n_val]), set(writers[n_train+n_val:])

train_writers, val_writers, test_writers = split_writers(df, seed=SEED)

print(f"Train: {len(train_writers)} writers   Val: {len(val_writers)}   Test: {len(test_writers)}")

# Map writer IDs to contiguous class indices (needed for cross-entropy)
sorted_train_writers = sorted(train_writers)
writer_to_class = {wid: idx for idx, wid in enumerate(sorted_train_writers)}
NUM_CLASSES = len(sorted_train_writers)
print(f"ArcFace num_classes (train writers): {NUM_CLASSES}")

---
## Section 3 — Datasets

**Training**: single genuine images with writer class label → ArcFace classification  
**Evaluation**: pairs (genuine vs genuine/forgery) → cosine similarity → AUC / EER

In [ ]:
class SignatureClassDataset(Dataset):
    """Single-image dataset for ArcFace: returns (image, class_idx)."""

    def __init__(self, df: pd.DataFrame, writer_to_class: dict,
                 writer_set: set, label_filter="genuine", transform=None):
        sub = df[(df["writer_id"].isin(writer_set)) & (df["label"] == label_filter)]
        self.records   = sub[["path", "writer_id"]].reset_index(drop=True)
        self.w2c       = writer_to_class
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records.iloc[idx]
        img = Image.open(row["path"]).convert("L")
        if self.transform:
            img = self.transform(img)
        cls = self.w2c[row["writer_id"]]
        return img, cls


class SiamesePairDataset(Dataset):
    """Pair dataset for evaluation: returns (img_a, img_b, label)."""

    def __init__(self, pairs_df: pd.DataFrame, transform=None):
        self.pairs     = pairs_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        row   = self.pairs.iloc[idx]
        img_a = Image.open(row["path_a"]).convert("L")
        img_b = Image.open(row["path_b"]).convert("L")
        if self.transform:
            img_a = self.transform(img_a)
            img_b = self.transform(img_b)
        return img_a, img_b, torch.tensor(row["label"], dtype=torch.float32)


IMG_SIZE = 224
BATCH    = 32
tfm      = get_default_transforms(IMG_SIZE)

train_cls_ds = SignatureClassDataset(df, writer_to_class, train_writers, transform=tfm)
print(f"Train classification dataset: {len(train_cls_ds)} genuine images, {NUM_CLASSES} classes")

train_loader = DataLoader(train_cls_ds, batch_size=BATCH, shuffle=True,
                          num_workers=0, pin_memory=DEVICE.type=="cuda")

x_sample, y_sample = next(iter(train_loader))
print("Batch shape:", x_sample.shape, "  labels:", y_sample[:8].tolist())

---
## Section 4 — ArcFace Loss

$$
L = -\log\frac{e^{s(\cos(\theta_{y_i}+m))}}{e^{s(\cos(\theta_{y_i}+m))} + \sum_{j \neq y_i} e^{s \cos\theta_j}}
$$

- $s$ = scale (controls logit magnitude, default 64)  
- $m$ = angular margin (default 0.5 rad ≈ 28.6°)  
- Weights $W$ and embeddings $z$ are both L2-normalised → dot product = cosine similarity

In [ ]:
class ArcFaceLoss(nn.Module):
    """
    ArcFace: Additive Angular Margin Loss.
    Holds a learnable weight matrix W of shape (num_classes, emb_dim).
    """

    def __init__(self, emb_dim: int, num_classes: int,
                 scale: float = 64.0, margin: float = 0.5):
        super().__init__()
        self.scale  = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.empty(num_classes, emb_dim))
        nn.init.xavier_uniform_(self.weight)

        # Pre-compute constants
        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        # Linearisation threshold at cos(pi - m)
        self.th    = math.cos(math.pi - margin)
        self.mm    = math.sin(math.pi - margin) * margin

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        # Normalise both embeddings and class weights
        emb_norm = F.normalize(embeddings, dim=1)
        w_norm   = F.normalize(self.weight,  dim=1)

        # Cosine similarity between each embedding and every class weight
        cos_theta = F.linear(emb_norm, w_norm).clamp(-1 + 1e-7, 1 - 1e-7)

        # cos(theta + m) = cos*cos_m - sin*sin_m
        sin_theta   = torch.sqrt(1.0 - cos_theta ** 2)
        cos_theta_m = cos_theta * self.cos_m - sin_theta * self.sin_m

        # Replace with linear approximation where theta > pi - m
        cos_theta_m = torch.where(
            cos_theta > self.th,
            cos_theta_m,
            cos_theta - self.mm
        )

        # Build output logits: margined logit for target class, normal for rest
        one_hot = torch.zeros_like(cos_theta)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1.0)
        output = (one_hot * cos_theta_m) + ((1.0 - one_hot) * cos_theta)
        output = output * self.scale

        return F.cross_entropy(output, labels.long())


# Quick sanity check
EMB_DIM = 128
_arc = ArcFaceLoss(EMB_DIM, NUM_CLASSES).to(DEVICE)
_emb = torch.randn(8, EMB_DIM).to(DEVICE)
_lbl = torch.randint(0, NUM_CLASSES, (8,)).to(DEVICE)
_loss = _arc(_emb, _lbl)
print("ArcFaceLoss sanity check:", _loss.item(), "  (scalar:", _loss.ndim == 0, ")")
del _arc, _emb, _lbl, _loss

---
## Section 5 — Model: Backbone + ArcFace Head

In [ ]:
# Backbone: SmallCNN (same as nb01 for fair comparison)
# ArcFaceLoss holds the classification head (W matrix)
backbone  = SmallCNN(emb_dim=EMB_DIM).to(DEVICE)
arc_head  = ArcFaceLoss(EMB_DIM, NUM_CLASSES, scale=32.0, margin=0.5).to(DEVICE)

# At inference: use backbone only, cosine similarity
n_backbone = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
n_head     = sum(p.numel() for p in arc_head.parameters()  if p.requires_grad)
print(f"Backbone params : {n_backbone:,}")
print(f"ArcFace W params: {n_head:,}   ({NUM_CLASSES} classes × {EMB_DIM} dim)")

# Test forward
with torch.no_grad():
    emb  = backbone(x_sample.to(DEVICE))
    loss = arc_head(emb, y_sample.to(DEVICE))
print(f"Test loss: {loss.item():.4f}   emb shape: {emb.shape}")

---
## Section 6 — Training

In [ ]:
set_seed(SEED)

backbone = SmallCNN(emb_dim=EMB_DIM).to(DEVICE)
arc_head = ArcFaceLoss(EMB_DIM, NUM_CLASSES, scale=32.0, margin=0.5).to(DEVICE)

optimizer = torch.optim.Adam(
    list(backbone.parameters()) + list(arc_head.parameters()),
    lr=1e-3
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

N_EPOCHS = 10
history  = {"loss": [], "acc": []}

for epoch in range(N_EPOCHS):
    backbone.train()
    arc_head.train()

    epoch_loss = 0.0
    correct    = 0
    total      = 0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS}", leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        emb  = backbone(imgs)
        loss = arc_head(emb, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Track train accuracy using cosine logits (without margin)
        with torch.no_grad():
            emb_n  = F.normalize(emb, dim=1)
            w_n    = F.normalize(arc_head.weight, dim=1)
            logits = F.linear(emb_n, w_n) * arc_head.scale
            preds  = logits.argmax(dim=1)
            correct += (preds == labels.long()).sum().item()
            total   += labels.size(0)

        epoch_loss += loss.item()

    scheduler.step()

    avg_loss = epoch_loss / len(train_loader)
    acc      = correct / total
    history["loss"].append(avg_loss)
    history["acc"].append(acc)

    print(f"Epoch {epoch+1:02d}/{N_EPOCHS}  loss={avg_loss:.4f}  train_acc={acc:.3f}  lr={scheduler.get_last_lr()[0]:.2e}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, N_EPOCHS+1), history["loss"], marker="o", color="royalblue")
ax1.set_title("ArcFace Training Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.grid(True)

ax2.plot(range(1, N_EPOCHS+1), history["acc"], marker="s", color="green")
ax2.set_title("Train Accuracy (writer classification)")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy"); ax2.grid(True)

plt.tight_layout()
plt.show()

---
## Section 7 — Evaluation Pairs

Build test pairs the same way as notebook 01 so results are directly comparable.

In [ ]:
def build_pools(df, writer_set):
    sub = df[df["writer_id"].isin(writer_set)]
    genuine_by_writer = {}
    forgery_by_writer = {}
    for wid, grp in sub.groupby("writer_id"):
        g = grp[grp["label"] == "genuine"]["path"].tolist()
        f = grp[grp["label"] == "forgery"]["path"].tolist()
        if g: genuine_by_writer[wid] = g
        if f: forgery_by_writer[wid] = f
    return genuine_by_writer, forgery_by_writer


def generate_pairs(df, writer_set, n_pairs=10_000, seed=1, neg_mix=0.8):
    genuine_by_writer, forgery_by_writer = build_pools(df, writer_set)
    writers           = sorted(genuine_by_writer.keys())
    writers_with_forg = sorted(set(genuine_by_writer) & set(forgery_by_writer))
    rng        = np.random.default_rng(seed)
    n_pos      = n_pairs // 2
    n_neg      = n_pairs - n_pos
    n_neg_same = int(round(n_neg * neg_mix))
    n_neg_cross= n_neg - n_neg_same
    rows = []
    for _ in range(n_pos):
        w = rng.choice(writers)
        g = genuine_by_writer[w]
        i, j = rng.choice(len(g), size=2, replace=False)
        rows.append({"path_a": g[i], "path_b": g[j], "label": 1,
                     "pair_type": "pos", "writer_a": w, "writer_b": w})
    for _ in range(n_neg_same):
        w = rng.choice(writers_with_forg)
        g, f = genuine_by_writer[w], forgery_by_writer[w]
        rows.append({"path_a": g[rng.integers(len(g))],
                     "path_b": f[rng.integers(len(f))],
                     "label": 0, "pair_type": "neg_same_writer",
                     "writer_a": w, "writer_b": w})
    for _ in range(n_neg_cross):
        w1, w2 = rng.choice(writers, size=2, replace=False)
        g1, g2 = genuine_by_writer[w1], genuine_by_writer[w2]
        rows.append({"path_a": g1[rng.integers(len(g1))],
                     "path_b": g2[rng.integers(len(g2))],
                     "label": 0, "pair_type": "neg_cross_writer",
                     "writer_a": w1, "writer_b": w2})
    return pd.DataFrame(rows).sample(frac=1.0, random_state=seed).reset_index(drop=True)


test_pairs = generate_pairs(df, test_writers, n_pairs=10_000, seed=3, neg_mix=0.8)
print(f"Test pairs: {len(test_pairs):,}")
print(test_pairs["label"].value_counts().to_string())

test_pair_ds     = SiamesePairDataset(test_pairs, transform=tfm)
test_pair_loader = DataLoader(test_pair_ds, batch_size=BATCH, shuffle=False,
                              num_workers=0, pin_memory=DEVICE.type=="cuda")

---
## Section 8 — Embedding Extraction & Metrics

Cosine similarity = dot product of L2-normalised embeddings (range −1 to 1, higher = more similar).

In [ ]:
backbone.eval()
all_sims   = []
all_labels = []

with torch.no_grad():
    for img_a, img_b, labels in tqdm(test_pair_loader, desc="Extracting embeddings"):
        e1 = F.normalize(backbone(img_a.to(DEVICE)), dim=1)
        e2 = F.normalize(backbone(img_b.to(DEVICE)), dim=1)
        # Cosine similarity per pair
        sim = (e1 * e2).sum(dim=1).cpu().numpy()
        all_sims.append(sim)
        all_labels.append(labels.numpy())

similarity = np.concatenate(all_sims)
y_true     = np.concatenate(all_labels).astype(int)

print("Cosine sim (pos):", similarity[y_true == 1].mean().round(4))
print("Cosine sim (neg):", similarity[y_true == 0].mean().round(4))

In [ ]:
results = compute_metrics(y_true, similarity)

print("\n=== ArcFace Test Metrics (CEDAR) ===")
for k, v in results.items():
    print(f"  {k:<20s}: {v:.4f}")

In [ ]:
fpr, tpr, _ = sk_metrics.roc_curve(y_true, similarity)

thresholds = np.linspace(similarity.min(), similarity.max(), 400)
far_arr, frr_arr = [], []
for t in thresholds:
    preds = (similarity >= t).astype(int)
    tp = int(((preds==1)&(y_true==1)).sum())
    fp = int(((preds==1)&(y_true==0)).sum())
    fn = int(((preds==0)&(y_true==1)).sum())
    tn = int(((preds==0)&(y_true==0)).sum())
    far_arr.append(fp/(fp+tn) if (fp+tn)>0 else 0.0)
    frr_arr.append(fn/(fn+tp) if (fn+tp)>0 else 0.0)

far_arr    = np.array(far_arr)
frr_arr    = np.array(frr_arr)
eer_thresh = results["eer_threshold"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC={results['auc']:.3f}")
axes[0].plot([0,1],[0,1],"k--")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("ROC — ArcFace"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(thresholds, far_arr, label="FAR", color="red")
axes[1].plot(thresholds, frr_arr, label="FRR", color="green")
axes[1].axvline(eer_thresh, linestyle="--", color="gray", label=f"EER thr={eer_thresh:.3f}")
axes[1].set_xlabel("Cosine similarity threshold")
axes[1].set_title(f"FAR/FRR  (EER={results['eer']:.3f})")
axes[1].legend(); axes[1].grid(True)

axes[2].hist(similarity[y_true==1], bins=50, alpha=0.6, color="green", label="Positive (same writer)")
axes[2].hist(similarity[y_true==0], bins=50, alpha=0.6, color="red",   label="Negative (forgery/cross)")
axes[2].set_xlabel("Cosine similarity")
axes[2].set_title("Similarity Distribution")
axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.show()

---
## Section 9 — Embedding Space Visualisation (PCA)

Extract embeddings for all test-set genuine signatures and project to 2D with PCA.  
Each colour = one writer. Tight clusters = good separation.

In [ ]:
# Build a single-image dataset for all test writers (genuine only)
test_writer_to_class = {wid: idx for idx, wid in enumerate(sorted(test_writers))}
test_cls_ds = SignatureClassDataset(df, test_writer_to_class, test_writers,
                                    label_filter="genuine", transform=tfm)
test_cls_loader = DataLoader(test_cls_ds, batch_size=BATCH, shuffle=False, num_workers=0)

backbone.eval()
all_embs    = []
all_wids    = []

with torch.no_grad():
    for imgs, cls_ids in tqdm(test_cls_loader, desc="Extracting test embeddings"):
        emb = F.normalize(backbone(imgs.to(DEVICE)), dim=1).cpu().numpy()
        all_embs.append(emb)
        all_wids.append(cls_ids.numpy())

all_embs = np.concatenate(all_embs)
all_wids = np.concatenate(all_wids)

print(f"Embeddings: {all_embs.shape}  |  Test writers: {len(test_writers)}")

# PCA to 2D
pca   = PCA(n_components=2, random_state=SEED)
emb2d = pca.fit_transform(all_embs)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")

fig, ax = plt.subplots(figsize=(9, 7))
cmap = plt.cm.get_cmap("tab20", len(test_writers))
for i, wid in enumerate(sorted(test_writer_to_class.keys())):
    mask = all_wids == test_writer_to_class[wid]
    ax.scatter(emb2d[mask, 0], emb2d[mask, 1],
               color=cmap(i), label=f"W{wid}", s=40, alpha=0.8)

ax.set_title("PCA of ArcFace Embeddings — Test Writers (genuine)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.legend(loc="upper right", fontsize=7, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 10 — Summary & Comparison with Contrastive Baseline

In [ ]:
from IPython.display import Markdown, display

rows_md = "\n".join(f"| {k:<20s} | {v:.4f} |" for k, v in results.items())

display(Markdown(f"""
### ArcFace Results — CEDAR (SmallCNN, {N_EPOCHS} epochs)

| Metric               | Value  |
|:---------------------|-------:|
{rows_md}

### Key differences from contrastive baseline (nb01)

| Aspect | Contrastive (nb01) | ArcFace (nb02) |
|--------|-------------------|----------------|
| Similarity metric | Euclidean distance | Cosine similarity |
| Forgeries in training | Yes | No (genuine only) |
| Pairs needed | Yes (40k) | No (single images) |
| Embedding norm | Unnormalised | L2-normalised |

### Notes
- Scale `s=32` used (lower than standard 64 due to small number of classes)
- Margin `m=0.5` rad (≈ 28.6°) — standard ArcFace setting
- CEDAR has only {NUM_CLASSES} training writers; ArcFace benefits more with more classes
"""))